In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns


app_name = "montage_pegasus-dss-1deg_node-16"
cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/"+app_name+"/"


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [4]:
from functions import *

In [5]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness

/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


In [6]:
use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness, DFGrepWorkflow, DFGrepWorkflow1

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "montage_pegasus-dss-1deg_node-16"

cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/"+app_name+"/"
os.makedirs(cp_dir, exist_ok=True)

condition_fn = None #

if app_name == "montage_pegasus-dss-1deg_node-16":
    filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-16/COMPACT/*.pfw.gz"

else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()



[INFO] [19:33:50] Initialized Client with 288 workers and link http://134.9.71.28:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:769]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [19:34:08] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:761]


In [7]:
def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))

def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 
    if "args" in json_object:
        if "exec_hash" in json_object["args"]:
            d["exec_hash"] = json_object["args"]["exec_hash"]

    if "name" in json_object:
        if (json_object["name"] in ["fwrite", "write","pwrite","fputs"]):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1
    return d

load_cols = {'size': "int64[pyarrow]", 'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]", 'exec_hash':"string[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}


In [8]:
analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)
# analyzer = DFAnalyzer(filename)

[INFO] [19:34:20] Created index for 14 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:428]
[INFO] [19:34:20] Total size of all files are <dask.bag.core.Item object at 0x1554ab40ad60> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:430]
[INFO] [19:34:20] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:433]
[INFO] [19:34:24] Loading 2013 batches out of 14 files and has 32884998 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:444]
[INFO] [19:35:36] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:511]
[INFO] [19:35:36] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:517]


# For Creating Graphs

In [ ]:
# if app_name == "montage_pegasus-dss-1deg_node-16":
if True:
    df1 = analyzer.events[~analyzer.events['fhash'].isna()] 
    df2 = analyzer.file_hash.reset_index()[["hash", "mount_point"]]
    df3 = analyzer.host_hash.reset_index()[['hhash', 'name']] 
    # # df3 = df3.rename(columns={'hash':'hhash'})
    result_df = df1.merge(df2, left_on="fhash", right_on="hash", how='left').drop(columns = ['hash_x', 'hash_y', 'mount_point_x']).rename(columns={'mount_point_y': 'mount_point'})
    result_df1 = result_df.merge(df3, on="hhash", how='left', suffixes=('_left', '_right')).rename(columns={'name_left': 'name', 'name_right':'hostname'})
    analyze_df = result_df1[['name','cat','size','ts','te','dur','trange','mount_point','hostname','fhash','pid','tid','prod','cons']] 
    analyze_df["mount_point"] = analyze_df["mount_point"].where(analyze_df["mount_point"].str.startswith("/"), "/p/lustre2")


In [7]:
fhash_dd = analyzer.file_hash.reset_index()[["hash","name"]]

In [8]:
from pathlib import PurePath
import dask.dataframe as dd
import pandas as pd
from pathlib import PurePath


# fhash_df , analyze_dd, mapping = unify_fhash_dask(fhash_dd, analyze_df)


In [35]:
fhash_df.query('hash =="decda2df349409a5"').compute()

,hash,name,basename,is_bare
12308,decda2df349409a5,./cposs2ukstu_red_001_001_area.fits,cposs2ukstu_red_001_001_area.fits,False
13808,decda2df349409a5,cposs2ukstu_red_001_001_area.fits,cposs2ukstu_red_001_001_area.fits,True
13968,decda2df349409a5,/p/lustre2/kogiou1/montage-workflow/scratch/ru...,cposs2ukstu_red_001_001_area.fits,False


In [9]:
changed_hash = [(k,v) for k,v in mapping.items() if k!=v]
len(changed_hash)

128

In [38]:
wf = DFGrepWorkflow(analyze_dd)
wf.select_events()

# lvl1_results = wf.get_wfGraph().compute()
# lvl2_results = wf.get_wfGraph(level=2).compute()
# lvl3_results = wf.get_wfGraph(level=3).compute()

lvl1_results = wf.get_wfGraph()
lvl2_results = wf.get_wfGraph(level=2)
lvl3_results = wf.get_wfGraph(level=3)

In [41]:
lvl2_results.drop_duplicates().compute().groupby("prod").count()

,pid,fhash,cons
prod,,,
0,252,252,252


In [11]:
lvl2_results.drop_duplicates().query("prod > 0").compute()

,pid,fhash,prod,cons


# Graphs

In [16]:
from functions import *
df3_unique_edges = lvl3_results.drop_duplicates()
df2_unique_edges = lvl2_results.drop_duplicates()
df1_unique_edges = lvl1_results.drop_duplicates()
# (A) Original: per (fhash, hostname)
l3_results = compute_io_metrics_host(df3_unique_edges, analyze_df, compute_result=False)

# (B) New: per (fhash, pid) using df2 (which has columns ['fhash','pid', ...])
l2_results = compute_io_metrics_pid(df2_unique_edges, analyze_df, compute_result=False)

l1_results = compute_io_metrics_pid(df1_unique_edges, analyze_df, compute_result=False)


In [18]:
l3_results.query("io_size > 0").compute()

,fhash,hostname,prod,cons,io_time,io_count,io_size,iops,bw
6,9.472957677855442e+17,corona229,0,1,440004,340174,4583158,0.773116,10.416173
7,1.7154868823274783e+19,corona229,0,1,449803,766305,48,1.703646,0.000107
8,4.668792765493391e+18,corona229,0,1,14132,1008,388,0.071327,0.027455
25,1.7778342866089847e+19,corona229,0,1,358972,241557,4583158,0.672913,12.767453
27,1.6134178151055628e+19,corona229,0,1,468,144,480,0.307692,1.025641
...,...,...,...,...,...,...,...,...,...
17922,1.4802300628565586e+19,corona238,0,1,2560,2,13,0.000781,0.005078
17924,4.1531914619509427e+18,corona244,0,1,29496,2,12,0.000068,0.000407
17925,1.185074726329838e+19,corona244,0,1,38610,2,13,0.000052,0.000337
17927,1.77470052982434e+17,corona237,0,1,34042,2,12,0.000059,0.000353


# Check

In [43]:
analyzer.string_hash.compute()

,name,pid,tid,hhash
hash,,,,
5549801160889696657,/usr/workspace/kogiou1/workflows/pegasus/insta...,145289,145289,10149931303662271596
8836348435689498274,pegasus-mpi-cluster,145289,145289,10149931303662271596
10196995424150485077,mProject,854649,854649,3813132488203668783
2944834342442738412,./mProject;-X;poss2ukstu_red_001_002.fits;ppos...,854649,854649,3813132488203668783
14284836729771958110,pegasus-kickstart,145707,145707,10149931303662271596
...,...,...,...,...
6524953605428228140,/p/lustre2/kogiou1/montage-workflow/scratch/ru...,855020,855020,3813132488203668783
3889787187376928799,mDiff;pposs2ukstu_ir_001_002.fits;pposs2ukstu_...,854927,854927,3813132488203668783
15809229069503248817,/p/lustre2/kogiou1/montage-workflow/scratch/ru...,145764,145764,10149931303662271596


In [47]:
(analyzer.events.astype(str) == "3889787187376928799").any().any().compute()


np.False_

In [44]:
analyzer.events.compute()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,app_io_time,total_time,fhash,phase,size,prod,cons,hash,mount_point,value
4,start,dftracer,0,145289,145289,10149931303662271596,1082504,1082504,0,<NA>,...,<NA>,0,<NA>,0,<NA>,0,1,<NA>,<NA>,<NA>
6,fopen,STDIO,0,145289,145289,10149931303662271596,1109588,1109664,76,<NA>,...,<NA>,0,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>
7,fclose,STDIO,0,145289,145289,10149931303662271596,1115706,1115736,30,<NA>,...,<NA>,0,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>
9,opendir,POSIX,0,145289,145289,10149931303662271596,1127024,1127085,61,<NA>,...,<NA>,61,7744662950880019,2,<NA>,0,1,<NA>,<NA>,<NA>
11,fopen,STDIO,0,145289,145289,10149931303662271596,1137742,1137880,138,<NA>,...,<NA>,0,7b5b92a310457d46,0,<NA>,0,1,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2545,close,POSIX,0,766912,766912,8347855584647769835,2002717823,2002717828,5,<NA>,...,<NA>,5,3b9d708549988072,2,0,0,1,<NA>,<NA>,<NA>
2546,close,POSIX,0,766912,766912,8347855584647769835,2002724512,2002724547,35,<NA>,...,<NA>,35,25be38de2a5705a4,2,0,0,1,<NA>,<NA>,<NA>
2547,close,POSIX,0,766912,766912,8347855584647769835,2003128172,2003128641,469,<NA>,...,<NA>,469,40cae248ad25b03b,2,0,0,1,<NA>,<NA>,<NA>
2548,close,POSIX,0,766912,766912,8347855584647769835,2003132342,2003132362,20,<NA>,...,<NA>,20,354f2255885644b1,2,0,0,1,<NA>,<NA>,<NA>


In [34]:
all_prods = analyzer.events.query("prod > 0")
all_prods.fhash.nunique().compute()

np.int64(196)

In [26]:
all_prods.groupby(["hhash","fhash"]).count().compute()

name   cat  type   pid   tid  \
hhash                fhash                                                  
10149931303662271596 2.4315893713484145e+18     1     1     1     1     1   
                     4.3465395197231416e+18     1     1     1     1     1   
3813132488203668783  1.2517304343964457e+19  5168  5168  5168  5168  5168   
                     1.654514717122134e+19   5168  5168  5168  5168  5168   
                     2.417988327361358e+18      1     1     1     1     1   
...                                           ...   ...   ...   ...   ...   
                     4.665533374328038e+17      1     1     1     1     1   
                     7.222294821943999e+17   1607  1607  1607  1607  1607   
                     9.360527667033715e+16   5185  5185  5185  5185  5185   
15706701541158148523 1.0620007849584425e+19    25    25    25    25    25   
                     7.515583715142639e+18     25    25    25    25    25   

                                               ts    te   dur  tinterval  \
hhash                fhash                                                 
10149931303662271596 2.4315893713484145e+18     1     1     1          0   
                     4.3465395197231416e+18     1     1     1          0   
3813132488203668783  1.2517304343964457e+19  5168  5168  5168          0   
                     1.654514717122134e+19   5168  5168  5168          0   
                     2.417988327361358e+18      1     1     1          0   
...                                           ...   ...   ...        ...   
                     4.665533374328038e+17      1     1     1          0   
                     7.222294821943999e+17   1607  1607  1607          0   
                     9.360527667033715e+16   5185  5185  5185          0   
15706701541158148523 1.0620007849584425e+19    25    25    25          0   
                     7.515583715142639e+18     25    25    25          0   

                                             trange  ...  io_time  \
hhash                fhash                           ...            
10149931303662271596 2.4315893713484145e+18       1  ...        1   
                     4.3465395197231416e+18       1  ...        1   
3813132488203668783  1.2517304343964457e+19    5168  ...        0   
                     1.654514717122134e+19     5168  ...        0   
                     2.417988327361358e+18        1  ...        1   
...                                             ...  ...      ...   
                     4.665533374328038e+17        1  ...        1   
                     7.222294821943999e+17     1607  ...        0   
                     9.360527667033715e+16     5185  ...        0   
15706701541158148523 1.0620007849584425e+19      25  ...        0   
                     7.515583715142639e+18       25  ...        0   

                                             app_io_time  total_time  phase  \
hhash                fhash                                                    
10149931303662271596 2.4315893713484145e+18            0           1      1   
                     4.3465395197231416e+18            0           1      1   
3813132488203668783  1.2517304343964457e+19            0        5168   5168   
                     1.654514717122134e+19             0        5168   5168   
                     2.417988327361358e+18             0           1      1   
...                                                  ...         ...    ...   
                     4.665533374328038e+17             0           1      1   
                     7.222294821943999e+17             0        1607   1607   
                     9.360527667033715e+16             0        5185   5185   
15706701541158148523 1.0620007849584425e+19            0          25     25   
                     7.515583715142639e+18             0          25     25   

                                             size  prod  cons  hash  \
hhash                fhash                               

In [35]:
all_prods.compute()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,app_io_time,total_time,fhash,phase,size,prod,cons,hash,mount_point,value
11295,write,POSIX,0,145289,145289,10149931303662271596,1463527070,1463527222,152,<NA>,...,<NA>,152,4.3465395197231416e+18,2,248,1,0,<NA>,<NA>,<NA>
11845,fwrite,STDIO,0,854649,854649,3813132488203668783,479271958,479273299,1341,<NA>,...,<NA>,0,5.57562982036309e+18,0,2880,1,0,<NA>,<NA>,<NA>
11846,fwrite,STDIO,0,854649,854649,3813132488203668783,479274180,479274181,1,<NA>,...,<NA>,0,5.57562982036309e+18,0,2880,1,0,<NA>,<NA>,<NA>
11847,fwrite,STDIO,0,854649,854649,3813132488203668783,479274987,479276071,1084,<NA>,...,<NA>,0,5.57562982036309e+18,0,17280,1,0,<NA>,<NA>,<NA>
11848,fwrite,STDIO,0,854649,854649,3813132488203668783,479276954,479277033,79,<NA>,...,<NA>,0,5.57562982036309e+18,0,2880,1,0,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10788,fwrite,STDIO,0,1587461,1587461,15706701541158148523,111492663,111492763,100,<NA>,...,<NA>,0,7.515583715142639e+18,0,1,1,0,<NA>,<NA>,<NA>
10789,fwrite,STDIO,0,1587461,1587461,15706701541158148523,111496961,111497048,87,<NA>,...,<NA>,0,7.515583715142639e+18,0,1,1,0,<NA>,<NA>,<NA>
10790,fwrite,STDIO,0,1587461,1587461,15706701541158148523,111501615,111501653,38,<NA>,...,<NA>,0,7.515583715142639e+18,0,1,1,0,<NA>,<NA>,<NA>
10791,fwrite,STDIO,0,1587461,1587461,15706701541158148523,111507091,111507189,98,<NA>,...,<NA>,0,7.515583715142639e+18,0,1,1,0,<NA>,<NA>,<NA>


In [47]:
count_df = all_prods.groupby(["pid", "hhash"]).size().reset_index()
count_df = count_df.rename(columns={0: "count"})  # rename the auto-generated column
filtered = count_df[count_df["count"] > 1000]
filtered_result = filtered.compute()

In [52]:
sum(filtered_result['count'])

416812

In [55]:
count_df.head()

,pid,hhash,count
0,145289,10149931303662271596,1
1,145296,10149931303662271596,1
2,854379,3813132488203668783,1
3,854399,3813132488203668783,1
4,854649,3813132488203668783,10266


In [8]:
analyzer.string_hash.head()

,name,pid,tid,hhash
hash,,,,
5549801160889696657,/usr/workspace/kogiou1/workflows/pegasus/insta...,145289,145289,10149931303662271596
8836348435689498274,pegasus-mpi-cluster,145289,145289,10149931303662271596
10196995424150485077,mProject,854649,854649,3813132488203668783
2944834342442738412,./mProject;-X;poss2ukstu_red_001_002.fits;ppos...,854649,854649,3813132488203668783
14284836729771958110,pegasus-kickstart,145707,145707,10149931303662271596


In [9]:
analyzer.events.columns

Index(['name', 'cat', 'type', 'pid', 'tid', 'hhash', 'ts', 'te', 'dur',
       'tinterval', 'trange', 'compute_time', 'io_time', 'app_io_time',
       'total_time', 'fhash', 'phase', 'size', 'prod', 'cons', 'exec_hash',
       'hash', 'mount_point', 'value'],
      dtype='object')

# Create App level Graph

In [9]:
start_events = analyzer.events.query("name == 'start'")
sh =  analyzer.string_hash.reset_index()
# start_events.head()

# Make sure both columns are same dtype
start_events = start_events.assign(exec_hash=start_events["exec_hash"].astype("string"))
sh = sh.assign(hash=sh["hash"].astype("string"))

# Select and rename first (keeps it as a Dask DataFrame)
sh_subset = sh[["hash", "name"]].rename(columns={"name": "app name"})

# Now perform merge safely
start_events = start_events.merge(sh_subset, left_on="exec_hash", right_on="hash", how="left")


prod_cons_events = analyzer.events.query("name in ['read', 'fgets', 'fread', 'fwrite', 'write', 'fputs']")
# prod_cons_events = analyzer.events

# grouped_prod_cons = (
#     prod_cons_events.groupby(['name','cat','pid','tid','hhash','fhash','prod','cons'])[["size"]]
#     .sum()
#     .reset_index()
#     .rename(columns={'size': 'total_size'})
# ).query('total_size > 0')

grouped_prod_cons = (
    prod_cons_events.groupby(['name','cat','pid','tid','hhash','fhash','prod','cons'])[["size"]]
    .sum()
    .reset_index()
    .rename(columns={'size': 'total_size'})
)

merged = grouped_prod_cons.merge(start_events[["pid","tid","app name"]], on=["pid","tid"], how="left")





In [8]:
merged.head()

,name,cat,pid,tid,hhash,fhash,prod,cons,total_size,app name
0,fgets,STDIO,3711240,3711240,2578120870981962427,ee12564debb781ba,0,1,0,pegasus-mpi-cluster
1,read,POSIX,3711240,3711240,2578120870981962427,5424335785880719,0,1,97510,pegasus-mpi-cluster
2,read,POSIX,3711240,3711240,2578120870981962427,00171042a7482487,0,1,7,pegasus-mpi-cluster
3,read,POSIX,3711240,3711240,2578120870981962427,016b315cc3f4f897,0,1,7,pegasus-mpi-cluster
4,read,POSIX,3711240,3711240,2578120870981962427,02310934914e0167,0,1,7,pegasus-mpi-cluster


In [10]:
fhash_dd = analyzer.file_hash.reset_index()[["hash","name"]]
fhash_df , merged_new, mapping = unify_fhash_dask(fhash_dd, merged)

In [11]:
def build_io_graph(df):
    G = nx.DiGraph()
    for _, row in df.iterrows():
        app = str(row["app name"])
        fhash = str(row["fhash"])
        weight = float(row.get("total_size", 1))

        if row["prod"] == 0: #if produced = 0 than app consumes the file (File -> App)
            # file → app
            G.add_edge(fhash, app, weight=weight)
        elif row["cons"] == 0: #if consumed = 0 than app 
            # app → file
            G.add_edge(app, fhash, weight=weight)
    return G

In [12]:
merged_new.head()

,name,cat,pid,tid,hhash,fhash,prod,cons,total_size,app name
0,fgets,STDIO,3711240,3711240,2578120870981962427,ee12564debb781ba,0,1,0,pegasus-mpi-cluster
1,read,POSIX,3711240,3711240,2578120870981962427,5424335785880719,0,1,97510,pegasus-mpi-cluster
2,read,POSIX,3711240,3711240,2578120870981962427,00171042a7482487,0,1,7,pegasus-mpi-cluster
3,read,POSIX,3711240,3711240,2578120870981962427,016b315cc3f4f897,0,1,7,pegasus-mpi-cluster
4,read,POSIX,3711240,3711240,2578120870981962427,02310934914e0167,0,1,7,pegasus-mpi-cluster


In [13]:
G = build_io_graph(merged_new.compute())

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 7507
Edges: 7804


In [14]:
sizes_wcc = [len(c) for c in nx.weakly_connected_components(G)]
print("Weakly connected component sizes:", sizes_wcc)

Weakly connected component sizes: [7505, 2]


In [44]:
edges = pd.DataFrame([(u, v, d.get("weight", 1)) for u, v, d in G.edges(data=True)],
                     columns=["src", "dest", "weight"])
edges.to_csv("graph_initial_allSizes.csv", index=False)


In [ ]:
target = "pegasus-mpi-cluster"
target = "flux-job"
comps = list(nx.weakly_connected_components(G))
target_comp = next((c for c in comps if target in c), set())

nodes = set().union(*[c for c in comps if c != target_comp and len(c) > 10])
G_filtered = G.subgraph(nodes).copy()


In [20]:
G_filtered1 = G_filtered.subgraph([n for n in G_filtered if "pegasus-mpi-cluster" not in n]).copy()


In [26]:
def remove_node_and_cleanup(G: nx.DiGraph, target: str) -> nx.DiGraph:
    """
    Remove a target node and all its edges from a directed graph.
    After removal, drop any nodes that become isolated (degree 0).
    """
    G_clean = G.copy()

    # 1. Remove the target node (and all incident edges)
    if target in G_clean:
        G_clean.remove_node(target)

    # 2. Find all nodes that are now isolated (no in- or out-edges)
    isolated_nodes = [n for n in G_clean.nodes() if G_clean.degree(n) == 0]

    # 3. Remove those isolated nodes
    G_clean.remove_nodes_from(isolated_nodes)

    return G_clean

G_filtered2 = remove_node_and_cleanup(G_filtered,"pegasus-mpi-cluster")


In [27]:
sizes_wcc1 = [len(c) for c in nx.weakly_connected_components(G_filtered2)]
print("Weakly connected component sizes:", sizes_wcc1)

Weakly connected component sizes: [5759]


In [31]:
for u, v, d in G_filtered2.edges(data=True):
    d["inv_size"] = 1.0 / d["weight"] if d.get("weight", 0) > 0 else 1e6


In [ ]:
# for u, v, data in G_filtered2.edges(data=True):
#     print(f"{u} → {v}  |  {data}")


ee12564debb781ba → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
327cdf8cf67adadf → mFitplane  |  {'weight': 15773906.0, 'inv_size': 6.339583867179124e-08}
d2e8a0080bd0e7d1 → mFitplane  |  {'weight': 15568082.0, 'inv_size': 6.423398849004007e-08}
d2e8a0080bd0e7d1 → mDiff  |  {'weight': 2880.0, 'inv_size': 0.00034722222222222224}
0001148b65c463d1 → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
000ee992967d0a99 → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
0020aef76b2dec2a → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
002631907671fbb3 → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
002b5ec0f667049c → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
002fa4c6fe56c462 → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
0032c626905fdab9 → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
00356e9dbf423c11 → pegasus-kickstart  |  {'weight': 0.0, 'inv_size': 1000000.0}
004823e27dd145

In [33]:
res = compute_node_metrics(G_filtered2)


unweightedComputation Done!
weightedComputation Done!


In [35]:
res_m = res[res["node"].str.lower().str.startswith("m")]
res_m

,node,betweenness_unweighted,betweenness_weighted,in_degree,out_degree
2,mFitplane,0.000000,0.000000,18,0
5384,mImgtbl,0.000020,0.000020,21,6
5392,mViewer,0.000014,0.000014,3,4
5482,mDiff,0.000075,0.000075,55,36
5545,mBackground,0.000098,0.000098,34,24
5565,mConcatFit,0.000032,0.000032,21,3
5630,mProject,0.000044,0.000044,17,24
5648,mAdd,0.000038,0.000038,34,6
5702,mBgModel,0.000040,0.000040,6,3


In [36]:
res.to_csv("/usr/workspace/pandey2/g_analyzer/Final_notebooks/TempDataflowResults/node16.csv")

In [41]:
total_sizes= merged_new.groupby("app name")["total_size"].sum()
total_sizes.compute()


app name
pegasus-mpi-cluster     148737682
mFitplane               216309436
mImgtbl                     36171
mViewer                1900866408
pegasus-kickstart      3196697692
mDiff                  4242771342
mBackground            2540020560
mConcatFit                    627
mProject               1453467498
mAdd                   1665543925
flux-job                        4
mBgModel                      168
Name: total_size, dtype: int64[pyarrow]

In [ ]:
import re
import networkx as nx

# 1) Betweenness (directed, unweighted)
bc = nx.betweenness_centrality(G_filtered, normalized=True, weight=None)

# 2) Heuristic: files look like hex hashes (8–64 hex chars); apps are everything else
is_file = re.compile(r"^[0-9a-fA-F]{8,64}$").match
files = [n for n in G_filtered if is_file(str(n))]
apps  = [n for n in G_filtered if not is_file(str(n))]

# 3) Top-5 by centrality
top5_files = sorted(files, key=lambda n: bc.get(n, 0.0), reverse=True)[:5]
top5_apps  = sorted(apps,  key=lambda n: bc.get(n, 0.0), reverse=True)[:]

# 4) (Optional) show with scores
top5_files_with_scores = [(n, bc[n]) for n in top5_files]
top5_apps_with_scores  = [(n, bc[n]) for n in top5_apps]

print("Top-5 FILES:", top5_files_with_scores)
print("Top-5 APPS: ", top5_apps_with_scores)

In [29]:
print(fhash_dd.query("hash in ['5424335785880719']").compute())

                  hash                                          name
5571  5424335785880719  /tmp/mv2-hwloc-f4uWt4b59-corona232-63170.xml


In [28]:
edges = pd.DataFrame([(u, v, d.get("weight", 1)) for u, v, d in G_filtered1.edges(data=True)],
                     columns=["src", "dest", "weight"])
edges.to_csv("graph-initial_pcallSizes.csv", index=False)

In [36]:
print(fhash_dd.query("hash in ['1e568f4f97e1cc88']").compute())
print(fhash_dd.query("hash in ['03bd3b31b8e81c2e']").compute())

print(fhash_dd.query("hash in ['9d6fa1e617187c4e']").compute())
print(fhash_dd.query("hash in ['8f064a6ef14247ac']").compute())
print(fhash_dd.query("hash in ['18e675ffad0203ee']").compute())

print(fhash_dd.query("hash in ['82968ccdc109e3aa']").compute())
print(fhash_dd.query("hash in ['9a4d8b8f9c356812']").compute())
print(fhash_dd.query("hash in ['84528a284b233cbf']").compute())
print(fhash_dd.query("hash in ['84528a284b233cbf']").compute())


print(fhash_dd.query("hash in ['4dafaf83f33fc229']").compute())



                   hash                                name
11478  1e568f4f97e1cc88  pposs2ukstu_blue_002_002_area.fits
                   hash                           name
11476  03bd3b31b8e81c2e  pposs2ukstu_blue_002_002.fits
                  hash               name
8201  9d6fa1e617187c4e  1-corrections.tbl
                   hash               name
11431  8f064a6ef14247ac  3-corrections.tbl
                  hash               name
5528  18e675ffad0203ee  2-corrections.tbl
                   hash                           name
11480  82968ccdc109e3aa  cposs2ukstu_blue_002_002.fits
                  hash        name
8199  9a4d8b8f9c356812  1-fits.tbl
                   hash                                               name
11218  84528a284b233cbf  /p/lustre2/kogiou1/montage-workflow/scratch/ru...
                   hash                                               name
11218  84528a284b233cbf  /p/lustre2/kogiou1/montage-workflow/scratch/ru...
                   hash             

In [ ]:
#1
start_events = analyzer.events.query("name == 'start'").compute()
# start_events[["exec_hash"]].head()

In [18]:
sh =  analyzer.string_hash.reset_index().compute()

In [13]:
start_events.head()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,total_time,fhash,phase,size,prod,cons,exec_hash,hash,mount_point,value
4,start,dftracer,0,145289,145289,10149931303662271596,1082504,1082504,0,<NA>,...,0,<NA>,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>
11307,start,dftracer,0,854649,854649,3813132488203668783,351791300,351791300,0,<NA>,...,0,<NA>,0,<NA>,0,1,8d83014ff0c5f855,<NA>,<NA>,<NA>
5734,start,dftracer,0,854379,854379,3813132488203668783,827580,827580,0,<NA>,...,0,<NA>,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>
1913,start,dftracer,0,145707,145707,10149931303662271596,1395588711,1395588711,0,<NA>,...,0,<NA>,0,<NA>,0,1,c63def5248829f5e,<NA>,<NA>,<NA>
5361,start,dftracer,0,855119,855119,3813132488203668783,1316761327,1316761327,0,<NA>,...,0,<NA>,0,<NA>,0,1,0f21a2b38f49d9c4,<NA>,<NA>,<NA>


In [11]:
sh.head()

,hash,name,pid,tid,hhash
0,4d04dac070e18191,/usr/workspace/kogiou1/workflows/pegasus/insta...,145289,145289,10149931303662271596
1,7aa10415d4d9e6a2,pegasus-mpi-cluster,145289,145289,10149931303662271596
2,28de273b67aba2ec,./mProject;-X;poss2ukstu_red_001_002.fits;ppos...,854649,854649,3813132488203668783
3,8d83014ff0c5f855,mProject,854649,854649,3813132488203668783
4,6e4922e65b019fc5,/p/lustre2/kogiou1/montage-workflow/scratch/ru...,145707,145707,10149931303662271596


In [19]:
import dask.dataframe as dd

# Make sure both columns are same dtype
start_events = start_events.assign(exec_hash=start_events["exec_hash"].astype("string"))
sh = sh.assign(hash=sh["hash"].astype("string"))

# Select and rename first (keeps it as a Dask DataFrame)
sh_subset = sh[["hash", "name"]].rename(columns={"name": "app name"})

# Now perform merge safely
start_events = start_events.merge(sh_subset, left_on="exec_hash", right_on="hash", how="left")



In [20]:
start_events

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,phase,size,prod,cons,exec_hash,hash_x,mount_point,value,hash_y,app name
0,start,dftracer,0,145289,145289,10149931303662271596,1082504,1082504,0,<NA>,...,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,7aa10415d4d9e6a2,pegasus-mpi-cluster
1,start,dftracer,0,854649,854649,3813132488203668783,351791300,351791300,0,<NA>,...,0,<NA>,0,1,8d83014ff0c5f855,<NA>,<NA>,<NA>,8d83014ff0c5f855,mProject
2,start,dftracer,0,854379,854379,3813132488203668783,827580,827580,0,<NA>,...,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,7aa10415d4d9e6a2,pegasus-mpi-cluster
3,start,dftracer,0,145707,145707,10149931303662271596,1395588711,1395588711,0,<NA>,...,0,<NA>,0,1,c63def5248829f5e,<NA>,<NA>,<NA>,c63def5248829f5e,pegasus-kickstart
4,start,dftracer,0,855119,855119,3813132488203668783,1316761327,1316761327,0,<NA>,...,0,<NA>,0,1,0f21a2b38f49d9c4,<NA>,<NA>,<NA>,0f21a2b38f49d9c4,mBackground
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,start,dftracer,0,2693038,2693038,12426538642911591065,1061785,1061785,0,<NA>,...,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,7aa10415d4d9e6a2,pegasus-mpi-cluster
917,start,dftracer,0,145674,145674,10149931303662271596,1276515813,1276515813,0,<NA>,...,0,<NA>,0,1,c63def5248829f5e,<NA>,<NA>,<NA>,c63def5248829f5e,pegasus-kickstart
918,start,dftracer,0,1013633,1013633,601656213817620015,1302811,1302811,0,<NA>,...,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,7aa10415d4d9e6a2,pegasus-mpi-cluster
919,start,dftracer,0,1662629,1662629,1496655337163968201,1014354,1014354,0,<NA>,...,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,7aa10415d4d9e6a2,pegasus-mpi-cluster


In [22]:
prod_cons_events = analyzer.events.query("name in ['read', 'fgets', 'fread', 'fwrite', 'write', 'fputs']")


In [25]:
len(prod_cons_events)

20335488

In [29]:
# Check the dtype
print(prod_cons_events["size"].dtype)

# Peek at a few values
# print(prod_cons_events["size"].head().compute())


int64[pyarrow]


In [35]:
import dask.dataframe as dd

# Make sure types are clean
df = prod_cons_events.assign(
    size = dd.to_numeric(prod_cons_events["size"], errors="coerce").fillna(0.0),
    pid  = dd.to_numeric(prod_cons_events["pid"],  errors="coerce").fillna(-1).astype("int64"),
    tid  = dd.to_numeric(prod_cons_events["tid"],  errors="coerce").fillna(-1).astype("int64"),
    name = prod_cons_events["name"].astype("string"),
    fhash= prod_cons_events["fhash"].astype("string"),
)

# Aggregate ONLY the 'size' column to avoid numeric_only issues
size_by_key = (
    df.groupby(["name", "fhash", "pid", "tid"])[["size"]]
      .sum()
      .reset_index()
      .rename(columns={"size": "total_size"})
)

# peek
size_by_key.head()
# or materialize:
# size_by_key.head().compute()


,name,fhash,pid,tid,total_size
0,fgets,0001148b65c463d1,145707,145707,0
1,fgets,000ee992967d0a99,145707,145707,0
2,fgets,0020aef76b2dec2a,145707,145707,0
3,fgets,002631907671fbb3,145707,145707,0
4,fgets,002b5ec0f667049c,145707,145707,0


In [36]:
size_by_key.head()

,name,fhash,pid,tid,total_size
0,fgets,0001148b65c463d1,145707,145707,0
1,fgets,000ee992967d0a99,145707,145707,0
2,fgets,0020aef76b2dec2a,145707,145707,0
3,fgets,002631907671fbb3,145707,145707,0
4,fgets,002b5ec0f667049c,145707,145707,0


In [ ]:
sh = analyzer.all_events.query("type == 3").drop_duplicates(subset=['value','name']).compute()

In [12]:
sh["name"] = sh["name"].astype(str).map(lambda x: x.split(";")[0])

In [13]:
sh.head()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,total_time,fhash,phase,size,prod,cons,exec_hash,hash,mount_point,value
2,/usr/workspace/kogiou1/workflows/pegasus/insta...,dftracer,3,145289,145289,10149931303662271596,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0,1,<NA>,5549801160889696657,<NA>,<NA>
3,pegasus-mpi-cluster,dftracer,3,145289,145289,10149931303662271596,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0,1,<NA>,8836348435689498274,<NA>,<NA>
11305,./mProject,dftracer,3,854649,854649,3813132488203668783,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0,1,<NA>,2944834342442738412,<NA>,<NA>
11306,mProject,dftracer,3,854649,854649,3813132488203668783,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0,1,<NA>,10196995424150485077,<NA>,<NA>
1911,/p/lustre2/kogiou1/montage-workflow/scratch/ru...,dftracer,3,145707,145707,10149931303662271596,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,0,1,<NA>,7946921390236606405,<NA>,<NA>


In [14]:
analyzer.events.head()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,total_time,fhash,phase,size,prod,cons,exec_hash,hash,mount_point,value
4,start,dftracer,0,145289,145289,10149931303662271596,1082504,1082504,0,<NA>,...,0,<NA>,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>
6,fopen,STDIO,0,145289,145289,10149931303662271596,1109588,1109664,76,<NA>,...,0,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>
7,fclose,STDIO,0,145289,145289,10149931303662271596,1115706,1115736,30,<NA>,...,0,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>
9,opendir,POSIX,0,145289,145289,10149931303662271596,1127024,1127085,61,<NA>,...,61,7744662950880019,2,<NA>,0,1,<NA>,<NA>,<NA>,<NA>
11,fopen,STDIO,0,145289,145289,10149931303662271596,1137742,1137880,138,<NA>,...,0,7b5b92a310457d46,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>


In [16]:
analyzer_events1 = analyzer.events.merge(
    sh[["pid", "name"]].rename(columns={"name": "app name"}),
    on="pid", how="left"
)

In [18]:
analyzer_events1.head()

,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,fhash,phase,size,prod,cons,exec_hash,hash,mount_point,value,app name
0,start,dftracer,0,145289,145289,10149931303662271596,1082504,1082504,0,<NA>,...,<NA>,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,/usr/workspace/kogiou1/workflows/pegasus/insta...
1,start,dftracer,0,145289,145289,10149931303662271596,1082504,1082504,0,<NA>,...,<NA>,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,pegasus-mpi-cluster
2,fopen,STDIO,0,145289,145289,10149931303662271596,1109588,1109664,76,<NA>,...,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>,/usr/workspace/kogiou1/workflows/pegasus/insta...
3,fopen,STDIO,0,145289,145289,10149931303662271596,1109588,1109664,76,<NA>,...,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>,pegasus-mpi-cluster
4,fclose,STDIO,0,145289,145289,10149931303662271596,1115706,1115736,30,<NA>,...,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>,/usr/workspace/kogiou1/workflows/pegasus/insta...


In [20]:
analyzer_events1[analyzer_events1["app name"].isna()].head()


,name,cat,type,pid,tid,hhash,ts,te,dur,tinterval,...,fhash,phase,size,prod,cons,exec_hash,hash,mount_point,value,app name
76602,start,dftracer,0,854379,854379,3813132488203668783,827580,827580,0,<NA>,...,<NA>,0,<NA>,0,1,7aa10415d4d9e6a2,<NA>,<NA>,<NA>,<NA>
76603,fopen,STDIO,0,854379,854379,3813132488203668783,840256,840409,153,<NA>,...,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>,<NA>
76604,fclose,STDIO,0,854379,854379,3813132488203668783,845864,845891,27,<NA>,...,e9f4101c64cb6b89,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>,<NA>
76605,opendir,POSIX,0,854379,854379,3813132488203668783,857021,857080,59,<NA>,...,7744662950880019,2,<NA>,0,1,<NA>,<NA>,<NA>,<NA>,<NA>
76606,fopen,STDIO,0,854379,854379,3813132488203668783,868906,868989,83,<NA>,...,7b5b92a310457d46,0,<NA>,0,1,<NA>,<NA>,<NA>,<NA>,<NA>
